# Fine-tune BGE-base-en-v1.5 for ArXiv Paper Similarity

Uses **CoSENTLoss** with hierarchical category distance and fuzzy multi-label similarity.

**Setup:**
1. Generate data locally: `python scripts/05_prepare_finetune_data.py --mode scored`
2. Zip: `cd ArXiv/data && zip -r finetune.zip finetune/`
3. Upload `finetune.zip` using the file upload button
4. Use GPU runtime (T4 or A100)

In [ ]:
# Install dependencies
!pip install -q sentence-transformers datasets

In [ ]:
# Upload finetune.zip via Colab file upload
# First, zip locally: cd ArXiv/data && zip -r finetune.zip finetune/
from google.colab import files
print("Upload finetune.zip...")
uploaded = files.upload()

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
import os

# Unzip uploaded data
!unzip -q finetune.zip -d /content/
DATA_DIR = '/content/finetune'
OUTPUT_DIR = '/content/bge-base-arxiv-finetuned'

# Verify data exists
for subdir in ['train', 'val', 'val_triplets']:
    path = os.path.join(DATA_DIR, subdir)
    assert os.path.exists(path), f"Missing: {path}"
    print(f"OK: {path}")

print("\nAll data found!")
!du -sh {DATA_DIR}

In [ ]:
import json
from datasets import load_from_disk

train_dataset = load_from_disk(os.path.join(DATA_DIR, 'train'))
val_dataset = load_from_disk(os.path.join(DATA_DIR, 'val'))
val_triplets = load_from_disk(os.path.join(DATA_DIR, 'val_triplets'))

# Detect mode from metadata
meta_path = os.path.join(DATA_DIR, 'metadata.json')
mode = 'pairs'
if os.path.exists(meta_path):
    with open(meta_path) as f:
        mode = json.load(f).get('mode', 'pairs')

print(f"Dataset mode: {mode}")
print(f"Train pairs:   {len(train_dataset)}")
print(f"Val pairs:     {len(val_dataset)}")
print(f"Val triplets:  {len(val_triplets)}")
print(f"Columns: {train_dataset.column_names}")

if mode == 'scored':
    scores = train_dataset['score']
    print(f"\nScore stats: min={min(scores):.3f}, max={max(scores):.3f}, mean={sum(scores)/len(scores):.3f}")
    print(f"Sample: s1={train_dataset[0]['sentence1'][:80]}...")
    print(f"        s2={train_dataset[0]['sentence2'][:80]}...")
    print(f"        score={train_dataset[0]['score']:.3f}")

In [ ]:
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss, CoSENTLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import TripletEvaluator

# Load model
model = SentenceTransformer('BAAI/bge-base-en-v1.5')
print(f"Model loaded. Embedding dim: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Setup evaluator
evaluator = TripletEvaluator(
    anchors=val_triplets['anchor'],
    positives=val_triplets['positive'],
    negatives=val_triplets['negative'],
    name='arxiv-val',
)

# Evaluate base model
print("Evaluating base model (before fine-tuning)...")
base_results = evaluator(model)
print(f"Base model accuracy: {base_results}")

In [ ]:
import torch
torch.cuda.empty_cache()

# Training config
# Adjust batch_size based on GPU:
#   T4 (16GB):  batch_size=64
#   A100 (40GB): batch_size=128

BATCH_SIZE = 64  # Change to 128 for A100
EPOCHS = 1
LR = 1e-5

# Select loss based on dataset mode
if mode == 'scored':
    loss = CoSENTLoss(model)
    print("Using CoSENTLoss (hierarchical + fuzzy similarity scores)")
else:
    loss = MultipleNegativesRankingLoss(model)
    print("Using MultipleNegativesRankingLoss (legacy)")

training_args_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    logging_steps=50,
)

if mode != 'scored':
    training_args_kwargs['batch_sampler'] = BatchSamplers.NO_DUPLICATES

training_args = SentenceTransformerTrainingArguments(**training_args_kwargs)

print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Steps per epoch: ~{len(train_dataset) // BATCH_SIZE}")
print(f"Total steps: ~{len(train_dataset) // BATCH_SIZE * EPOCHS}")

In [ ]:
from datetime import datetime

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
    evaluator=evaluator,
)

start = datetime.now()
trainer.train()
duration = (datetime.now() - start).total_seconds()

print(f"\nTraining completed in {duration:.0f}s ({duration/60:.1f} min)")

In [ ]:
# Evaluate fine-tuned model
print("Evaluating fine-tuned model...")
finetuned_results = evaluator(model)
print(f"\nBase model:      {base_results}")
print(f"Fine-tuned model: {finetuned_results}")

base_acc = base_results['arxiv-val_cosine_accuracy']
ft_acc = finetuned_results['arxiv-val_cosine_accuracy']
print(f"\nImprovement: {base_acc:.4f} -> {ft_acc:.4f} ({ft_acc - base_acc:+.4f})")

In [ ]:
# Save and download fine-tuned model
FINAL_DIR = os.path.join(OUTPUT_DIR, 'final')
model.save_pretrained(FINAL_DIR)
print(f"Model saved to: {FINAL_DIR}")
!du -sh {FINAL_DIR}

# Zip for download
!cd {OUTPUT_DIR} && zip -r /content/bge-base-arxiv-finetuned.zip final/
print("\nModel zipped. Downloading...")

from google.colab import files
files.download('/content/bge-base-arxiv-finetuned.zip')